In [88]:
import pandas as pd
import requests
import logging
import yaml
import os
import time
from dotenv import load_dotenv

In [89]:
with open('config_log.yaml', 'r') as f:
    config = yaml.safe_load(f)
if config['logging']['turn_on']:
    logging.basicConfig(filename=config['logging']['file'], level=config['logging']['level'], format='%(asctime)s (%(name)s) [%(levelname)s] - %(message)s')
    logger_API = logging.getLogger('API')
else:
    logger_API = logging.getLogger('API')
    logger_API.addHandler(logging.NullHandler())

logger_API.info("Старт парсинга")

Далее из .env загружаю нужные ключи (чтобы не палились хехе)

In [90]:
load_dotenv()

True

In [91]:
CLIENT_ID = os.getenv('CLIENT_ID')
CLIENT_SECRET = os.getenv('CLIENT_SECRET')

Создаем функцию для обращения и выдачи access_token

In [92]:
def get_access_token(client_id, client_secret):
    logger_API.info("Запрашиваем token")
    url = 'https://id.twitch.tv/oauth2/token'
    params = {
        'client_id': client_id, 
        'client_secret': client_secret, 
        'grant_type': 'client_credentials'
    }
    try:
        response = requests.post(url, params=params)
        response.raise_for_status()
        logger_API.info("Token получен успешно") 
        token_data = response.json()
        return token_data['access_token']
    except Exception as e:
        logger_API.error(f"Ошибка при получении token: {e}")
        raise
access_token = get_access_token(CLIENT_ID, CLIENT_SECRET)


Создаем заголовки для дальнейших запросов

In [93]:
headers = {
    'Client-ID': CLIENT_ID, 
    'Authorization': f'Bearer {access_token}'}

Создаем "шаблонную" функцию, чтобы постоянно ее не переписывать, далее будем менять эндпоинты и содержимое запроса при вызове функции

In [94]:
def request_sample(endpoint, query, headers):
    url = f'https://api.igdb.com/v4/{endpoint}'
    response = requests.post(url=url, headers=headers, data=query)
    response.raise_for_status()
    return response.json()

Теперь запросы

Сначала создадим справочник жанров игр

In [95]:
genres_query = '''
fields id, name;
limit 500;
'''
genres = request_sample('genres', genres_query, headers)
df_genres = pd.DataFrame(genres)
df_genres.head()

,id,name
0,2,Point-and-click
1,4,Fighting
2,5,Shooter
3,7,Music
4,8,Platform


In [96]:
df_genres.shape

(23, 2)

In [97]:
df_genres.to_csv('data/genres.csv', index=False)

Теперь справочник платформ, на которых можно играть в игрушечки

In [98]:
platforms_query = '''
fields id, name, abbreviation;
limit 500;
'''
platforms = request_sample('platforms', platforms_query, headers)
df_platfroms = pd.DataFrame(platforms)
df_platfroms.head()

,id,name,abbreviation
0,376,Epoch Super Cassette Vision,NaN
1,510,e-Reader / Card-e Reader,NaN
2,12,Xbox 360,X360
3,32,Sega Saturn,Saturn
4,62,Atari Jaguar,Jaguar


In [99]:
df_platfroms.shape

(220, 3)

In [100]:
df_platfroms.to_csv('data/platforms.csv', index=False)

Теперь справочник режимов игр

In [101]:
modes_query = '''
fields id, name;
limit 500;
'''
game_modes = request_sample('game_modes', modes_query, headers)
df_game_modes = pd.DataFrame(game_modes)
df_game_modes.head()

,id,name
0,1,Single player
1,2,Multiplayer
2,3,Co-operative
3,4,Split screen
4,5,Massively Multiplayer Online (MMO)


In [102]:
df_game_modes.shape

(6, 2)

In [103]:
df_game_modes.to_csv('data/game_modes.csv', index=False)

Справочник тематик игр (отличается от жанров тем, что жанр - то, каким образом происходит игра (рпг, шутер, рогалик и так далее), а тематика - общее состояние вокруг, атмосфера (хоррор, фантастика и так далее))

In [104]:
themes_query = '''
fields id, name;
limit 500;
'''
themes = request_sample('themes', themes_query, headers)
df_themes = pd.DataFrame(themes)
df_themes.head()

,id,name
0,31,Drama
1,32,Non-fiction
2,33,Sandbox
3,34,Educational
4,35,Kids


In [105]:
df_themes.shape

(22, 2)

In [106]:
df_themes.to_csv('data/themes.csv', index=False)

Рассмотрим еще game_status и game_category справочники

In [107]:
game_statuses_query = '''
fields created_at, status, updated_at;
limit 500;
'''
game_statuses = request_sample('game_statuses', game_statuses_query, headers)
df_game_statuses = pd.DataFrame(game_statuses)
df_game_statuses

,id,status,created_at,updated_at
0,0,Released,1738540800,1738540800
1,2,Alpha,1738540800,1738540800
2,3,Beta,1738540800,1738540800
3,4,Early Access,1738540800,1738540800
4,5,Offline,1738540800,1738540800
5,6,Cancelled,1738540800,1738540800
6,7,Rumored,1738540800,1738540800
7,8,Delisted,1738540800,1738540800


In [108]:
df_game_statuses.shape

(8, 4)

In [109]:
df_game_statuses.to_csv('data/game_statuses.csv', index=False)

In [110]:
game_types_query = '''
fields created_at, type, updated_at;
limit 500;
'''
game_types = request_sample('game_types', game_types_query, headers)
df_game_types = pd.DataFrame(game_types)
df_game_types

,id,type,created_at,updated_at
0,3,Bundle,1743587713,1743587713
1,1,DLC,1743587713,1743587713
2,6,Episode,1743587713,1743587713
3,4,Standalone Expansion,1743587713,1743587713
4,5,Mod,1743587713,1743587713
5,8,Remake,1743587713,1743587713
6,11,Port,1743587713,1743587713
7,7,Season,1743587713,1743587713
8,12,Fork,1743587713,1743587713
9,14,Update,1743587713,1743587713


In [111]:
df_game_types.shape

(15, 4)

In [112]:
df_game_types.to_csv('data/game_types.csv', index=False)

## Основной цикл для сбора данных по играм

In [113]:
df = pd.DataFrame()
offset = 0
games_query_sample = '''
fields id, name, rating, aggregated_rating, aggregated_rating_count, total_rating, total_rating_count, hypes, first_release_date, genres, platforms, game_modes, 
themes, game_status, game_type, age_ratings, storyline, summary;
where total_rating_count >= 5;
sort total_rating desc;
limit 500;
offset {offset};
'''
while len(df) < 10000:
    try:
        logger_API.info(f"Запрос offset={offset}, собрано: {len(df)}")
        games_query = games_query_sample.format(offset=offset)
        buffer = request_sample('games', games_query, headers)
        df_buffer = pd.DataFrame(buffer)
        df = pd.concat([df, df_buffer], ignore_index=True)
        df = df.drop_duplicates(subset='id').reset_index(drop=True)
    except Exception as e:
        logger_API.error(f"Ошибка при запросе offset={offset}: {e}")
        continue
    offset += 500
    time.sleep(0.5)
logger_API.info(f"Сбор данных завершён, всего игр: {len(df)}")
df.head()

,id,first_release_date,game_modes,genres,name,platforms,rating,summary,themes,total_rating,total_rating_count,game_type,storyline,age_ratings,game_status,aggregated_rating,aggregated_rating_count,hypes
0,376092,1.761869e+09,[1],[32],Silly Survivors,[6],100.0,"Silly Survivors - a casual, pared-down rogueli...",[1],100.0,6,0,NaN,NaN,NaN,NaN,NaN,NaN
1,330539,1.744934e+09,"[2, 3, 5]","[5, 12, 13]",Oxide: Survival Island,"[34, 39]",100.0,OXIDE is a mobile multiplayer survival simulat...,"[21, 23, 33, 38]",100.0,9,0,"Here you are, alone on the abandoned island, w...",NaN,NaN,NaN,NaN,NaN
2,328386,1.736986e+09,NaN,NaN,Rooftop Rascal: The Claus Cat,"[48, 167]",100.0,"In ""Rooftop Rascal: The Claus Cat,"" take contr...",NaN,100.0,7,0,NaN,[196045],NaN,NaN,NaN,NaN
3,269473,1.696550e+09,"[1, 2, 3]",[32],Resonite,"[3, 6, 163]",100.0,Enter a novel digital universe with infinite p...,"[28, 33, 34]",100.0,10,0,NaN,NaN,4.0,NaN,NaN,NaN
4,65755,1.311898e+09,[1],"[2, 31]",Trailer Park King,[12],100.0,"He's a gamer, a bootlegger, a photographer and...",[27],100.0,47,0,NaN,NaN,NaN,NaN,NaN,NaN


In [114]:
df.shape

(10000, 18)

In [115]:
df.isna().sum()

id                            0
first_release_date           13
game_modes                  183
genres                       24
name                          0
platforms                     0
rating                       84
summary                      19
themes                      686
total_rating                  0
total_rating_count            0
game_type                     0
storyline                  6377
age_ratings                1814
game_status                9648
aggregated_rating          3979
aggregated_rating_count    3979
hypes                      7602
dtype: int64

In [116]:
df.to_csv('data/games.csv', index=False, encoding='utf-8')